# 🔒 啟動私人 AI（教師隱私資料處理）

這個筆記本會在**你自己 Google 帳號的私人 Colab 主機**上，安裝本地 AI（Strata＋Qwen 模型）與 OpenCode 網頁介面。
資料只在這台主機內處理，**不會送到任何 AI 公司**；用完刪除執行階段，主機就會銷毀。

### 使用步驟
1. 確認右上角登入的是**有付費 Colab 運算單元**的 Google 帳號。
2. 選單「執行階段 → 變更執行階段類型」，確認硬體加速器是 **G4**（本筆記本已預設）。
3. 選單「執行階段 → 全部執行」。若跳出「這個筆記本不是由 Google 編寫的」，按「仍要執行」。
4. 等待約 10–15 分鐘，第 ⑤ 格下方就會出現對話框，直接打字即可。
5. 用完請執行最後一格「清理」，再按「執行階段 → 中斷連線並刪除執行階段」。

### ⚠️ 隱私提醒
- **不要使用 Colab 裡的 Gemini 功能**處理學生資料（Gemini 的對話可能被人工審閱）。
- 建議上傳前先把學生姓名等資料換成代號。
- 使用期間請保持這個分頁開著。

In [ ]:
#@title ① 檢查主機（GPU、記憶體、磁碟）
import shutil, subprocess

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit('❌ 沒有偵測到 GPU。請到「執行階段 → 變更執行階段類型」選擇 G4 後重新執行。')
gpu_name, vram_mib = [s.strip() for s in gpu.stdout.strip().splitlines()[0].split(',')]
vram_gb = int(vram_mib) / 1024
ram_gb = int(open('/proc/meminfo').readline().split()[1]) / 1024 / 1024
disk_gb = shutil.disk_usage('/content').free / 1024**3

print(f'GPU：{gpu_name}（{vram_gb:.0f} GB）')
print(f'系統記憶體：{ram_gb:.0f} GB')
print(f'可用磁碟：{disk_gb:.0f} GB')

# Qwen3.8 IQ3_S 需要約 55 GB（顯示卡＋系統記憶體合計）
MODEL = 'iq3_s' if vram_gb + ram_gb >= 70 else 'iq2_xs'
print(f'➡ 使用模型：Qwen3.8-Flash-Next {MODEL.upper()}')
if disk_gb < 90:
    print('⚠️ 可用磁碟少於 90 GB，模型可能下載不完。')

In [ ]:
#@title ② 準備核心腳本（自動產生，請勿修改）
import os
os.makedirs('/content/core', exist_ok=True)
FILES = {}
FILES['setup_strata.sh'] = r'''#!/usr/bin/env bash
# 安裝 Strata 並在背景啟動模型，等到模型載入完成才結束。
# 用法：bash setup_strata.sh [模型代號，預設 iq3_s] [最長等待分鐘數，預設 120]
set -euo pipefail

MODEL="${1:-iq3_s}"
MAX_WAIT_MIN="${2:-120}"
# 上下文長度：模型上限 256K；G4（96GB 顯卡＋177GB 記憶體）以 --plan 確認可行
CONTEXT="${STRATA_CONTEXT:-256k}"
# Colab 自己的服務佔用 8080，Strata 改用 8181
STRATA_PORT="${STRATA_PORT:-8181}"
LOG_DIR=/content/logs
HEALTH_URL="http://127.0.0.1:${STRATA_PORT}/health"
mkdir -p "$LOG_DIR"
export PATH="$HOME/.local/bin:/usr/local/bin:$PATH"

is_loaded() {
  curl -fs --max-time 5 "$HEALTH_URL" 2>/dev/null | grep -Eq '"loaded"[[:space:]]*:[[:space:]]*true'
}

if is_loaded; then
  echo "✅ Strata 已在執行，模型已載入。"
  exit 0
fi

if ! command -v strata >/dev/null 2>&1; then
  echo "▶ 安裝 Strata…"
  curl -fsSL https://github.com/Niko1221/Strata/releases/latest/download/install.sh -o /tmp/strata_install.sh
  sh /tmp/strata_install.sh
  # 安裝程式會把 strata 放進 PATH，但 Colab 的非互動 shell 不一定讀得到，補建一個連結
  if ! command -v strata >/dev/null 2>&1; then
    found=$(find "$HOME/.local" -maxdepth 5 -type f -name strata -perm -u+x 2>/dev/null | head -n1 || true)
    if [ -z "$found" ]; then
      echo "❌ 找不到 strata 執行檔，安裝可能失敗。" >&2
      exit 1
    fi
    ln -sf "$found" /usr/local/bin/strata
  fi
fi
# Python 核心（portal.py）的 PATH 沒有 ~/.local/bin，統一連到 /usr/local/bin
ln -sf "$(command -v strata)" /usr/local/bin/strata 2>/dev/null || true
strata --version

echo "▶ 啟動 Strata（模型 $MODEL，上下文 $CONTEXT）。第一次會下載約 55–76 GB，請耐心等候…"
nohup strata launch --model "$MODEL" --context "$CONTEXT" --port "$STRATA_PORT" --yes --no-tui > "$LOG_DIR/strata.log" 2>&1 &

start=$(date +%s)
while ! is_loaded; do
  elapsed=$(( ($(date +%s) - start) / 60 ))
  if [ "$elapsed" -ge "$MAX_WAIT_MIN" ]; then
    echo "❌ 等待超過 $MAX_WAIT_MIN 分鐘仍未載入，最後幾行紀錄：" >&2
    tail -n 20 "$LOG_DIR/strata.log" >&2
    exit 1
  fi
  last=$(tail -n 1 "$LOG_DIR/strata.log" 2>/dev/null | tr -d '\r' | cut -c1-120)
  echo "  ⏳ 已等待 ${elapsed} 分鐘｜${last}"
  sleep 30
done

echo "✅ 模型載入完成，總共 $(( ($(date +%s) - start) / 60 )) 分鐘。"
'''
FILES['setup_opencode.sh'] = r'''#!/usr/bin/env bash
# 安裝 OpenCode（agent 與網頁介面）。
set -euo pipefail
# 固定實測過的版本：入口網址格式（/<目錄>/session）依版本而異
OPENCODE_VERSION="${OPENCODE_VERSION:-1.19.4}"
export PATH="$HOME/.opencode/bin:$HOME/.local/bin:/usr/local/bin:$PATH"

if ! command -v opencode >/dev/null 2>&1; then
  echo "▶ 安裝 OpenCode $OPENCODE_VERSION…"
  curl -fsSL https://opencode.ai/install -o /tmp/opencode_install.sh
  bash /tmp/opencode_install.sh --version "$OPENCODE_VERSION"
  for candidate in "$HOME/.opencode/bin/opencode" "$HOME/.local/bin/opencode"; do
    if [ -x "$candidate" ]; then
      ln -sf "$candidate" /usr/local/bin/opencode
      break
    fi
  done
fi

if ! command -v opencode >/dev/null 2>&1; then
  echo "❌ 找不到 opencode 執行檔，安裝可能失敗。" >&2
  exit 1
fi
echo "✅ OpenCode $(opencode --version)"
'''
FILES['setup_tools.sh'] = r'''#!/usr/bin/env bash
# 預先安裝文件處理工具整合包，避免老師使用時 OpenCode 才要求安裝（例如 OCR 繁中語言包）。
# 用法：bash setup_tools.sh            （前景安裝）
#       bash setup_tools.sh --background（背景安裝，與模型下載同時進行）
#       bash setup_tools.sh --wait      （等待背景安裝完成，再跑 selfcheck_tools.py 確認真的能用）
set -uo pipefail

LOG_DIR=/content/logs
DONE=/content/logs/tools.done
mkdir -p "$LOG_DIR"

APT_PACKAGES=(
  tesseract-ocr tesseract-ocr-chi-tra tesseract-ocr-chi-sim tesseract-ocr-eng  # OCR（含繁中）
  poppler-utils                                                               # pdftotext、pdftoppm
  pandoc                                                                      # Markdown ↔ Word 等轉檔
  libreoffice-writer libreoffice-calc libreoffice-impress                     # docx／xlsx／pptx 轉 PDF
  fonts-noto-cjk                                                              # 中文字型，轉 PDF 不會變方塊
  graphviz                                                                    # 流程圖、組織圖
  ghostscript unpaper pngquant                                                # ocrmypdf 需要的工具（ocrmypdf 本身用 pip 裝）
  fonts-arphic-ukai fonts-arphic-uming                                        # 楷書、明體：代替公文常用的標楷體、新細明體
)
PIP_PACKAGES=(
  python-docx openpyxl python-pptx pdfplumber pymupdf pytesseract pdf2image  # 文件與 OCR
  xlsxwriter tabulate seaborn plotly graphviz                                 # 表格與圖表（已裝的會略過）
  docxtpl docxcompose                                                         # Word 範本合併列印、合併多份 Word
  pypdf pyzipper msoffcrypto-tool xlrd                                        # PDF 合併／加密、AES 加密壓縮、有密碼的 Office 檔、舊版 .xls
  opencc-python-reimplemented jieba                                           # 簡轉繁（台灣用語）、中文斷詞
  ocrmypdf                                                                    # 掃描 PDF 轉可搜尋文字（apt 版是給系統 Python 3.10 的，Colab 的 python3 較新會無法執行）
)
# pandas、numpy、matplotlib、Pillow、OpenCV 是 Colab 內建，不重裝（由 selfcheck_tools.py 確認）

# 學校文件常指定標楷體、新細明體、微軟正黑體（Windows 字型，Linux 沒有），轉 PDF 時改用最接近的字型
configure_fontconfig() {
  cat > /etc/fonts/local.conf <<'EOF'
<?xml version="1.0"?>
<!DOCTYPE fontconfig SYSTEM "fonts.dtd">
<fontconfig>
  <alias><family>標楷體</family><prefer><family>AR PL UKai TW</family></prefer></alias>
  <alias><family>DFKai-SB</family><prefer><family>AR PL UKai TW</family></prefer></alias>
  <alias><family>BiauKai</family><prefer><family>AR PL UKai TW</family></prefer></alias>
  <alias><family>新細明體</family><prefer><family>AR PL UMing TW</family></prefer></alias>
  <alias><family>PMingLiU</family><prefer><family>AR PL UMing TW</family></prefer></alias>
  <alias><family>細明體</family><prefer><family>AR PL UMing TW</family></prefer></alias>
  <alias><family>MingLiU</family><prefer><family>AR PL UMing TW</family></prefer></alias>
  <alias><family>微軟正黑體</family><prefer><family>Noto Sans CJK TC</family></prefer></alias>
  <alias><family>Microsoft JhengHei</family><prefer><family>Noto Sans CJK TC</family></prefer></alias>
</fontconfig>
EOF
  fc-cache -f >/dev/null 2>&1
}

# 圖表中文字型：matplotlib 預設字型沒有中文，會畫成方塊
configure_matplotlib() {
  local cfg cache
  cfg=$(python3 -c 'import matplotlib; print(matplotlib.get_configdir())') || return 1
  cache=$(python3 -c 'import matplotlib; print(matplotlib.get_cachedir())') || return 1
  rm -rf "$cache"/fontlist-*.json  # 讓 matplotlib 重新掃描，找到剛裝的 Noto CJK
  # 用 matplotlib 實際認得的字型名稱（.ttc 只會登記其中一個，常是 JP），避免寫了不存在的名稱一直跳警告
  local family
  family=$(python3 - <<'PY'
from matplotlib import font_manager as fm
names = {f.name for f in fm.fontManager.ttflist if "CJK" in f.name}
print(next((n for n in ("Noto Sans CJK TC", "Noto Sans CJK JP") if n in names), sorted(names)[0] if names else ""))
PY
)
  [ -n "$family" ] || { echo "⚠️ matplotlib 找不到中文字型" >&2; return 1; }
  mkdir -p "$cfg"
  cat > "$cfg/matplotlibrc" <<EOF
font.family: sans-serif
font.sans-serif: $family, DejaVu Sans
axes.unicode_minus: False
EOF
}

install_all() {
  rm -f "$DONE"
  export DEBIAN_FRONTEND=noninteractive
  apt-get update -qq
  apt-get install -y -qq --no-install-recommends "${APT_PACKAGES[@]}"
  pip install -q "${PIP_PACKAGES[@]}"
  configure_fontconfig
  configure_matplotlib
  tesseract --list-langs 2>/dev/null | grep -q chi_tra && touch "$DONE"
}

case "${1:-}" in
  --background)
    nohup bash "$0" > "$LOG_DIR/tools.log" 2>&1 &
    echo "▶ 文件工具整合包在背景安裝中（OCR 繁中、PDF、Word／Excel／PPT、表格與圖表、合併列印、加密、公文字型）…"
    ;;
  --wait)
    for _ in $(seq 1 120); do
      if [ -f "$DONE" ]; then
        echo "✅ 文件工具整合包已安裝，開始自我檢查（約 30 秒）…"
        python3 "$(dirname "$0")/selfcheck_tools.py" 2>&1 | tee "$LOG_DIR/selfcheck.log"
        exit 0  # 自我檢查沒過只警告，不擋 AI 啟動
      fi
      sleep 5
    done
    echo "⚠️ 文件工具安裝未完成，最後幾行紀錄：" >&2
    tail -n 15 "$LOG_DIR/tools.log" >&2
    exit 1
    ;;
  *)
    install_all
    [ -f "$DONE" ] || { echo "❌ 文件工具安裝失敗" >&2; exit 1; }
    ;;
esac
'''
FILES['selfcheck_tools.py'] = r'''"""開機自我檢查：確認文件工具「能用」，不只是「裝了」。

在暫存資料夾產生測試檔（不碰工作區、不碰學生資料）：
- 中文圖表：用 matplotlib／seaborn 畫圖，攔截「字型缺字」警告（中文變方塊時一定會出現）；
  graphviz 用 fc-match 確認找得到中文字型。
- OCR：把一行中文畫成乾淨的圖片，用繁中 OCR 單行模式讀回來。
- 產生含表格與原生圖表的 Word／Excel／PowerPoint → 轉成 PDF → 從 PDF 讀回中文；
  另檢查 xlsxwriter、Markdown 表格、plotly 離線 HTML、合併列印、PDF／壓縮檔／Office 加密、
  掃描 PDF 轉文字、公文字型對應、簡轉繁與斷詞、pandoc。
任何一項失敗只顯示警告，不會擋住 AI 啟動。
"""
import json
import os
import shutil
import subprocess
import sys
import tempfile
import warnings

TEXT = "學生成績統計"
SCORES = [("國文", 80), ("數學", 72), ("英文", 90)]
CJK_FILE = "NotoSansCJK"  # fonts-noto-cjk 的字型檔名
results = []


def check(name):
    def wrap(fn):
        try:
            detail = fn()
            results.append((True, name, detail or ""))
        except Exception as e:  # 自我檢查要把每一項都跑完，不因單項失敗中斷
            results.append((False, name, f"{type(e).__name__}: {e}"))
        return fn
    return wrap


def compact(s):
    return "".join(s.split())


def save_without_missing_glyphs(fig, path):
    """存圖時攔截 matplotlib 的「Glyph … missing from font」警告：有缺字就代表中文會變方塊。"""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        fig.savefig(path)
    missing = [str(w.message) for w in caught if "missing from font" in str(w.message)]
    if missing:
        raise RuntimeError(f"中文缺字（會變方塊）：{missing[0][:80]}")


work = tempfile.mkdtemp(prefix="tools_selfcheck_")
os.chdir(work)


@check("內建套件（pandas、numpy、matplotlib、Pillow、OpenCV）")
def _():
    import cv2, matplotlib, numpy, pandas, PIL  # noqa: F401
    return f"pandas {pandas.__version__}、OpenCV {cv2.__version__}"


@check("matplotlib 中文圖表（不缺字）")
def _():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    font = font_manager.findfont(font_manager.FontProperties(family=["sans-serif"]))
    if CJK_FILE not in font:
        raise RuntimeError(f"預設字型不是中文字型：{os.path.basename(font)}")
    fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
    ax.bar([s for s, _ in SCORES], [v for _, v in SCORES])
    ax.set_title(TEXT, fontsize=28)
    save_without_missing_glyphs(fig, "chart.png")
    plt.close(fig)
    return f"字型：{os.path.basename(font)}"


@check("seaborn 中文統計圖（不缺字）")
def _():
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(font=plt.rcParams["font.sans-serif"][0])  # 與 AGENTS.md 寫給 AI 的用法一致
    try:
        ax = sns.barplot(x=[s for s, _ in SCORES], y=[v for _, v in SCORES])
        ax.set_title(TEXT, fontsize=28)
        save_without_missing_glyphs(ax.figure, "seaborn.png")
        plt.close(ax.figure)
    finally:
        sns.reset_orig()


@check("OCR 繁中（單行文字讀回）")
def _():
    import pytesseract
    from matplotlib import font_manager
    from PIL import Image, ImageDraw, ImageFont
    font_path = font_manager.findfont(font_manager.FontProperties(family=["sans-serif"]))
    img = Image.new("L", (900, 160), 255)
    ImageDraw.Draw(img).text((30, 30), TEXT, fill=0, font=ImageFont.truetype(font_path, 80))
    img.save("ocr.png")
    text = compact(pytesseract.image_to_string(img, lang="chi_tra", config="--psm 7"))
    hit = sum(ch in text for ch in TEXT)
    if hit < len(TEXT) - 1:  # 容許錯一個字
        raise RuntimeError(f"只讀對 {hit}/{len(TEXT)} 字：{text[:30]}")
    return f"讀對 {hit}/{len(TEXT)} 字"


@check("Word 表格＋圖片、Excel／PowerPoint 原生圖表")
def _():
    import docx, openpyxl, pptx
    from docx.shared import Inches as DocxInches
    from openpyxl.chart import BarChart, Reference
    from pptx.chart.data import CategoryChartData
    from pptx.enum.chart import XL_CHART_TYPE
    from pptx.util import Inches
    d = docx.Document()
    d.add_heading(TEXT, 1)
    d.add_paragraph("國文 80 分，數學 72 分。")
    table = d.add_table(rows=1, cols=2)
    table.style = "Table Grid"
    table.rows[0].cells[0].text, table.rows[0].cells[1].text = "科目", "分數"
    for subject, score in SCORES:
        row = table.add_row().cells
        row[0].text, row[1].text = subject, str(score)
    if os.path.exists("chart.png"):
        d.add_picture("chart.png", width=DocxInches(4))
    d.save("doc.docx")

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.append(["科目", "分數"])
    for row in SCORES:
        ws.append(list(row))
    chart = BarChart()
    chart.title = TEXT
    chart.add_data(Reference(ws, min_col=2, min_row=1, max_row=len(SCORES) + 1), titles_from_data=True)
    chart.set_categories(Reference(ws, min_col=1, min_row=2, max_row=len(SCORES) + 1))
    ws.add_chart(chart, "D2")
    wb.save("sheet.xlsx")

    p = pptx.Presentation()
    slide = p.slides.add_slide(p.slide_layouts[5])  # 只有標題的版面
    slide.shapes.title.text = TEXT
    data = CategoryChartData()
    data.categories = [s for s, _ in SCORES]
    data.add_series("分數", [v for _, v in SCORES])
    slide.shapes.add_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, Inches(1), Inches(1.5), Inches(8), Inches(5), data)
    p.save("slides.pptx")

    import pandas
    if pandas.read_excel("sheet.xlsx").iloc[0]["分數"] != 80:
        raise RuntimeError("Excel 讀回的內容不對")
    if len(openpyxl.load_workbook("sheet.xlsx").active._charts) != 1:
        raise RuntimeError("Excel 圖表沒有存進去")
    if len(docx.Document("doc.docx").tables[0].rows) != len(SCORES) + 1:
        raise RuntimeError("Word 表格讀回的列數不對")
    if not any(s.has_chart for s in pptx.Presentation("slides.pptx").slides[0].shapes):
        raise RuntimeError("PowerPoint 圖表沒有存進去")
    return "寫入並讀回確認"


@check("xlsxwriter 圖表、Markdown 表格（tabulate）")
def _():
    import pandas
    import xlsxwriter
    wb = xlsxwriter.Workbook("xw.xlsx")
    ws = wb.add_worksheet()
    ws.write_row(0, 0, ["科目", "分數"])
    for i, (subject, score) in enumerate(SCORES, start=1):
        ws.write_row(i, 0, [subject, score])
    chart = wb.add_chart({"type": "column"})
    chart.add_series({"categories": ["Sheet1", 1, 0, len(SCORES), 0],
                      "values": ["Sheet1", 1, 1, len(SCORES), 1], "name": TEXT})
    ws.insert_chart("D2", chart)
    wb.close()
    md = pandas.DataFrame(SCORES, columns=["科目", "分數"]).to_markdown(index=False)
    if "| 國文" not in md:
        raise RuntimeError("to_markdown 輸出不對")


@check("plotly 互動圖表（離線 HTML）")
def _():
    import plotly.express as px
    fig = px.bar(x=[s for s, _ in SCORES], y=[v for _, v in SCORES], title=TEXT)
    fig.write_html("plotly.html", include_plotlyjs=True)
    html = open("plotly.html", encoding="utf-8").read()
    if len(html) < 1_000_000:  # 內含完整 plotly.js（約 4–5 MB）才能離線打開
        raise RuntimeError("HTML 沒有內含 plotly.js，離線會打不開")
    if TEXT not in html and json.dumps(TEXT)[1:-1] not in html:  # plotly 會把中文寫成 \uXXXX
        raise RuntimeError("HTML 裡找不到圖表標題")
    return f"{len(html) // 1_000_000} MB，離線可開"


@check("graphviz 中文流程圖")
def _():
    import graphviz
    font = "Noto Sans CJK TC"
    matched = subprocess.run(["fc-match", "-f", "%{file}", font], capture_output=True, text=True).stdout
    if CJK_FILE not in matched:
        raise RuntimeError(f"系統找不到中文字型「{font}」：{matched}")
    g = graphviz.Digraph(format="png", graph_attr={"fontname": font},
                         node_attr={"fontname": font, "shape": "box"})
    g.edge("蒐集資料", TEXT)
    g.edge(TEXT, "撰寫報告")
    g.render("flow", cleanup=True)
    return f"字型：{os.path.basename(matched)}"


@check("轉成 PDF（LibreOffice）")
def _():
    subprocess.run(["soffice", "--headless", "--convert-to", "pdf",
                    "doc.docx", "sheet.xlsx", "slides.pptx"],
                   check=True, capture_output=True, timeout=240)
    pdfs = [n for n in os.listdir() if n.endswith(".pdf")]
    if len(pdfs) < 3:
        raise RuntimeError(f"只產生 {len(pdfs)} 個 PDF（應有 3 個）")
    return "Word、Excel、PowerPoint 各轉出一個 PDF"


@check("從 PDF 讀回中文（pdfplumber、PyMuPDF、poppler）")
def _():
    import pdfplumber, pymupdf
    with pdfplumber.open("doc.pdf") as pdf:
        t1 = compact(pdf.pages[0].extract_text() or "")
    t2 = compact(pymupdf.open("doc.pdf")[0].get_text())
    t3 = compact(subprocess.run(["pdftotext", "doc.pdf", "-"], capture_output=True,
                                text=True, check=True).stdout)
    for name, t in (("pdfplumber", t1), ("PyMuPDF", t2), ("pdftotext", t3)):
        if TEXT not in t:
            raise RuntimeError(f"{name} 讀不到中文（字型可能沒嵌入）")
    subprocess.run(["pdftoppm", "-png", "-r", "50", "-f", "1", "-l", "1", "doc.pdf", "page"],
                   check=True, capture_output=True)
    return "三種方式都讀到中文，也能轉成圖片"


@check("合併列印（docxtpl＋docxcompose）")
def _():
    import docx
    from docxcompose.composer import Composer
    from docxtpl import DocxTemplate
    tpl = docx.Document()
    tpl.add_paragraph("{{ 姓名 }} 同學的國文成績：{{ 分數 }} 分")
    tpl.save("tpl.docx")
    letters = []
    for name, score in (("學生A", 80), ("學生B", 72)):
        t = DocxTemplate("tpl.docx")
        t.render({"姓名": name, "分數": score})
        t.save(f"letter_{name}.docx")
        letters.append(f"letter_{name}.docx")
    merged = Composer(docx.Document(letters[0]))
    merged.append(docx.Document(letters[1]))
    merged.save("letters.docx")
    text = "".join(p.text for p in docx.Document("letters.docx").paragraphs)
    if "學生A 同學" not in text or "學生B 同學" not in text:
        raise RuntimeError(f"合併結果不對：{text[:40]}")
    return "2 份通知單產生並合併"


@check("加密與舊格式（pypdf、pyzipper、msoffcrypto、xlrd）")
def _():
    import io
    import msoffcrypto
    import openpyxl
    import pyzipper
    import xlrd  # noqa: F401  pandas 讀 .xls 需要
    from msoffcrypto.format.ooxml import OOXMLFile
    from pypdf import PdfReader, PdfWriter
    w = PdfWriter()
    w.add_blank_page(width=200, height=200)
    w.add_blank_page(width=200, height=200)
    w.encrypt("test-pw", algorithm="AES-256")
    w.write("locked.pdf")
    r = PdfReader("locked.pdf")
    if not r.is_encrypted or not r.decrypt("test-pw") or len(r.pages) != 2:
        raise RuntimeError("PDF 加密／解密不對")
    with pyzipper.AESZipFile("locked.zip", "w", compression=pyzipper.ZIP_DEFLATED,
                             encryption=pyzipper.WZ_AES) as z:
        z.setpassword(b"test-pw")
        z.writestr("成績.txt", "國文 80")
    with pyzipper.AESZipFile("locked.zip") as z:
        z.setpassword(b"test-pw")
        if z.read("成績.txt").decode() != "國文 80":
            raise RuntimeError("加密壓縮讀回不對")
    wb = openpyxl.Workbook()
    wb.active["A1"] = "國文"
    wb.save("plain.xlsx")
    with open("plain.xlsx", "rb") as fin, open("locked.xlsx", "wb") as fout:
        OOXMLFile(fin).encrypt("test-pw", fout)
    office = msoffcrypto.OfficeFile(open("locked.xlsx", "rb"))
    office.load_key(password="test-pw")
    buf = io.BytesIO()
    office.decrypt(buf)
    if openpyxl.load_workbook(buf).active["A1"].value != "國文":
        raise RuntimeError("有密碼的 Excel 解開後內容不對")
    return "PDF、zip、Excel 加密後都能用密碼解開"


@check("掃描 PDF 轉可搜尋（ocrmypdf）")
def _():
    from PIL import Image
    if not os.path.exists("ocr.png"):
        raise RuntimeError("缺少 OCR 測試圖（上面的 OCR 項目沒有跑成功）")
    Image.open("ocr.png").convert("RGB").save("scan.pdf", resolution=150)
    run = subprocess.run(["ocrmypdf", "-l", "chi_tra", "--output-type", "pdf", "scan.pdf", "scan_ocr.pdf"],
                         capture_output=True, text=True, timeout=180)
    if run.returncode != 0:
        raise RuntimeError(f"結束碼 {run.returncode}：{run.stderr.strip()[-200:]}")
    text = compact(subprocess.run(["pdftotext", "scan_ocr.pdf", "-"], capture_output=True,
                                  text=True, check=True).stdout)
    hit = sum(ch in text for ch in TEXT)
    if hit < len(TEXT) - 1:
        raise RuntimeError(f"轉出的 PDF 只讀到 {hit}/{len(TEXT)} 字：{text[:30]}")
    return f"讀到 {hit}/{len(TEXT)} 字"


@check("公文字型對應（標楷體→楷書、新細明體→明體）")
def _():
    want = {"標楷體": "ukai", "新細明體": "uming", "微軟正黑體": "NotoSansCJK"}
    got = {}
    for family, expect in want.items():
        got[family] = subprocess.run(["fc-match", "-f", "%{file}", family],
                                     capture_output=True, text=True).stdout
        if expect not in got[family]:
            raise RuntimeError(f"{family} 對應到 {os.path.basename(got[family]) or '（無）'}")
    return "、".join(f"{k}→{os.path.basename(v)}" for k, v in got.items())


@check("簡轉繁、中文斷詞（opencc、jieba）")
def _():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # jieba 匯入時的 pkg_resources 過時警告，無害
        import jieba
    from opencc import OpenCC
    jieba.setLogLevel(60)  # 不顯示載入字典的訊息
    converted = OpenCC("s2twp").convert("软件信息")
    if converted != "軟體資訊":
        raise RuntimeError(f"簡轉繁結果：{converted}")
    words = jieba.lcut("學生的閱讀理解需要加強")
    if len(words) < 3:
        raise RuntimeError(f"斷詞結果：{words}")
    return f"软件信息→{converted}；斷詞 {len(words)} 個"


@check("Markdown 轉 Word（pandoc）")
def _():
    with open("note.md", "w", encoding="utf-8") as f:
        f.write(f"# {TEXT}\n\n- 國文 80 分\n")
    subprocess.run(["pandoc", "note.md", "-o", "note.docx"], check=True, capture_output=True)
    import docx
    if TEXT not in docx.Document("note.docx").paragraphs[0].text:
        raise RuntimeError("轉出的 Word 內容不對")
    return ""


os.chdir("/")
shutil.rmtree(work, ignore_errors=True)

failed = [r for r in results if not r[0]]
print("🔎 文件工具自我檢查：")
for ok, name, detail in results:
    print(f"  {'✅' if ok else '❌'} {name}" + (f"（{detail}）" if detail else ""))
if failed:
    print(f"⚠️ {len(failed)} 項沒通過，AI 仍可使用，但相關功能可能要請老師換別的方式處理。")
    sys.exit(1)
print("✅ 全部通過：Word、Excel、PowerPoint、PDF、OCR、表格與圖表、合併列印、加密、公文字型都能用。")
'''
FILES['portal.py'] = r'''"""產生 OpenCode 設定、啟動網頁入口、關閉與清理。

筆記本與 agent 都只呼叫這裡的函式，確保兩種模式行為一致。
本檔不會讀取或列出工作區裡的檔案內容（學生資料）。
"""
import base64
import json
import os
import secrets
import shutil
import signal
import subprocess
import time
import urllib.error
import urllib.request

# Colab 自己的服務佔用 8080，Strata 改用 8181（與 setup_strata.sh 一致）
STRATA_URL = f"http://127.0.0.1:{os.environ.get('STRATA_PORT', '8181')}"
PORTAL_PORT = 4096
# 老師的所有檔案（上傳的、AI 產出的）都放在這一個資料夾，也是入口預設開啟的專案
WORKSPACE = "/content/工作區"
LOG_DIR = "/content/logs"
CONFIG_PATH = os.path.expanduser("~/.config/opencode/opencode.json")
OPENCODE_DATA = os.path.expanduser("~/.local/share/opencode")
PID_FILE = os.path.join(LOG_DIR, "opencode.pid")
COMPACT_AT = 0.8  # 用到上下文 80% 就自動壓縮，避免爆掉
# 思考強度：Strata 接受 app 的 reasoning effort（off／low／medium／high），
# OpenCode 的 variants 讓老師在介面切換；off 最快，high 適合難題
REASONING_LEVELS = ["off", "low", "medium", "high"]

AGENT_RULES = """# 工作規則

你是協助老師處理教學資料的助理，在一台私人主機上執行。

- 所有檔案都在目前這個資料夾：老師上傳的檔案在這裡，你產出的檔案也一律存在這裡（不要另建子資料夾，除非老師要求）。
- 檔案可能含學生個資：不得連網、不得把內容傳到任何外部服務。
- 已預先安裝好的工具，直接使用，**不要再安裝其他套件或下載任何東西**；缺少工具時告訴老師：
  - OCR：`tesseract`（含繁中 chi_tra、簡中 chi_sim、英文 eng），Python `pytesseract`
  - PDF：`pdftotext`／`pdftoppm`（poppler），Python `pdfplumber`、`pymupdf`（PyMuPDF）
  - Word／Excel／PowerPoint：Python `docx`、`openpyxl`、`pptx`；轉檔用 `pandoc`、`soffice --headless --convert-to`（LibreOffice）
  - 表格資料分析：Python `pandas`、`numpy`（可直接讀寫 Excel）
  - 圖片處理：Python `PIL`（Pillow）、`cv2`（OpenCV）；你也可以直接看圖（模型支援圖片輸入）
- 表格與圖表的做法（中文字型都已設定好）：
  - Word 表格：`docx` 的 `add_table`（可設 `style = "Table Grid"`）；Word 裡的圖表：先用 matplotlib 存成 PNG，再 `add_picture`
  - Excel 表格與圖表：`openpyxl.chart` 或 `xlsxwriter`，做成 Excel 原生圖表，老師之後還能在 Excel 裡改
  - PowerPoint 圖表：`pptx` 的 `shapes.add_chart`（原生圖表，可再編輯）
  - 統計圖存成圖片：`matplotlib`（不必指定字型）；用 `seaborn` 時要寫 `sns.set_theme(font=plt.rcParams["font.sans-serif"][0])`（沿用已設定的中文字型），否則中文會變方塊
  - 互動圖表：`plotly` 存成 HTML，一定要用 `write_html(..., include_plotlyjs=True)`，才能離線打開
  - 流程圖、組織圖：`graphviz`，節點與圖都要設 `fontname="Noto Sans CJK TC"`
  - 在對話中顯示表格：用 Markdown 表格，或 `pandas` 的 `to_markdown()`
- 老師常見的文書工作：
  - 合併列印（每位學生一份通知單、成績單、IEP）：`docxtpl` 用 Word 範本（`{{ 姓名 }}`）逐一產生；要合成一個檔用 `docxcompose`
  - PDF 合併、分割、加密碼：`pypdf`；要加密壓縮整批檔案用 `pyzipper`（AES）。密碼由老師提供，不要自己設定或寫進檔名
  - 學校寄來有密碼的 Excel／Word：`msoffcrypto-tool`（要先請老師提供密碼）
  - 舊版 `.xls` 用 `pandas.read_excel`（已裝 xlrd）；舊版 `.doc`／`.ppt` 先用 `soffice --headless --convert-to docx`（或 pptx）轉檔再處理
  - 掃描的 PDF：`ocrmypdf -l chi_tra+eng 輸入.pdf 輸出.pdf` 轉成可搜尋、可複製文字；手寫或表格很複雜時，直接看圖通常比 OCR 準
  - 文件指定標楷體／新細明體／微軟正黑體時照常寫進檔案；轉 PDF 會自動用楷書／明體／黑體代替
  - 簡體轉繁體（台灣用語）：`opencc`（`OpenCC("s2twp")`）；所有產出一律用繁體中文
  - 輔導紀錄、問卷開放題的關鍵字與斷詞：`jieba`
  - 去識別化：先用代號取代姓名，身分證字號（如 `[A-Z][12]\\d{8}`）、電話、生日、地址用正規表示式找出來遮蔽，完成後列出替換了哪些欄位讓老師確認
- 用繁體中文回答；產出 IEP 等正式文件時保持專業、具體、可評量。
- 不確定的地方標註「待老師確認」，不要自行編造學生資料。
"""


def _get_json(path):
    with urllib.request.urlopen(STRATA_URL + path, timeout=10) as resp:
        return json.loads(resp.read().decode("utf-8"))


def strata_model():
    """回傳 Strata 目前提供的 (模型 ID, 上下文長度)。"""
    models = _get_json("/v1/models").get("data", [])
    if not models:
        raise RuntimeError("Strata 沒有回報任何模型，請確認模型已載入。")
    # 以 Strata 實際載入的上下文為準（setup_strata.sh 預設要 256K）；讀不到時保守用 65536
    return models[0]["id"], models[0].get("meta", {}).get("n_ctx", 65536)


def write_config(context=None, output=16384):
    """寫入 OpenCode 全域設定：只連本機 Strata、禁止分享與連網；上下文長度跟 Strata 一致。

    自動壓縮：OpenCode 1.19.4（session/overflow.ts）在 已用 token ≥ limit.input − compaction.reserved
    時壓縮，所以 input 設滿、reserved 設 20%，就是用到 80% 時壓縮。
    """
    model_id, n_ctx = strata_model()
    context = context or n_ctx
    config = {
        "$schema": "https://opencode.ai/config.json",
        "share": "disabled",
        "autoupdate": False,
        "model": f"strata/{model_id}",
        "small_model": f"strata/{model_id}",
        "provider": {
            "strata": {
                "npm": "@ai-sdk/openai-compatible",
                "name": "Strata（私人主機上的 AI）",
                "options": {"baseURL": STRATA_URL + "/v1", "apiKey": "local"},
                "models": {
                    model_id: {
                        "name": model_id,
                        "limit": {"context": context, "input": context, "output": output},
                        "modalities": {"input": ["text", "image"], "output": ["text"]},
                        # @ai-sdk/openai-compatible 會把 reasoningEffort 送成 reasoning_effort
                        "variants": {level: {"reasoningEffort": level} for level in REASONING_LEVELS},
                    }
                },
            }
        },
        "compaction": {"auto": True, "prune": True, "reserved": int(context * (1 - COMPACT_AT))},
        "permission": {"webfetch": "deny"},
    }
    os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    return model_id


def reasoning_check():
    """用一題簡單算術確認 Strata 真的依思考強度調整：off 不該有思考內容，high 應該有。"""
    def thinks(level):
        body = {"model": "local", "max_tokens": 300, "reasoning_effort": level,
                "messages": [{"role": "user", "content": "1+1 等於多少？只回答數字。"}]}
        req = urllib.request.Request(STRATA_URL + "/v1/chat/completions", data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=120) as resp:
            msg = json.loads(resp.read().decode("utf-8"))["choices"][0]["message"]
        thought = msg.get("reasoning_content") or msg.get("reasoning") or ""
        return bool(thought.strip()) or "<think>" in (msg.get("content") or "")

    try:
        off, high = thinks("off"), thinks("high")
    except Exception as e:  # 檢查失敗不影響入口
        return f"⚠️ 無法確認思考強度（{type(e).__name__}: {e}）"
    if high and not off:
        return "✅ 思考強度可切換：off 不思考、high 會思考（入口可選 " + "／".join(REASONING_LEVELS) + "）"
    return f"⚠️ 思考強度可能沒有生效（off 有思考：{off}，high 有思考：{high}）"


def entry_path(directory=WORKSPACE):
    """入口直接開到「工作區」的新對話，省掉「新增專案」步驟。

    OpenCode 1.19.4 網頁版路由 `/:dir/session/:id?`，dir 為目錄的 URL-safe base64（去掉 =），
    見 packages/app/src/utils/session-route.ts 的 legacySessionHref。
    """
    encoded = base64.urlsafe_b64encode(directory.encode("utf-8")).decode("ascii").rstrip("=")
    return f"/{encoded}/session"


def _port_open(port):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=3)
        return True
    except urllib.error.HTTPError:
        return True  # 401 等回應代表服務已經起來
    except OSError:
        return False


def start(port=PORTAL_PORT, wait_seconds=120, use_password=False):
    """啟動 OpenCode 網頁介面，回傳 (使用者名稱, 本次密碼)；不設密碼時回傳 (None, None)。

    預設不設密碼：入口內嵌在 Colab 筆記本裡，只有開著這本筆記本、登入同一個
    Google 帳號的瀏覽器連得到（換瀏覽器實測為 404）；內嵌畫面也無法跳出帳密框。
    """
    os.makedirs(WORKSPACE, exist_ok=True)
    os.makedirs(LOG_DIR, exist_ok=True)
    rules = os.path.join(WORKSPACE, "AGENTS.md")
    if not os.path.exists(rules):
        with open(rules, "w", encoding="utf-8") as f:
            f.write(AGENT_RULES)

    stop_portal()
    env = dict(os.environ)
    env.pop("OPENCODE_SERVER_PASSWORD", None)
    password = secrets.token_urlsafe(9) if use_password else None
    if password:
        env["OPENCODE_SERVER_PASSWORD"] = password
    log = open(os.path.join(LOG_DIR, "opencode.log"), "w")
    # Colab 的連接埠代理是從主機內網位址連進來，只綁 127.0.0.1 會 404；
    # 綁 0.0.0.0 仍只能經 Google 驗證的代理到達，另有密碼保護。Strata 維持只綁 127.0.0.1。
    proc = subprocess.Popen(
        ["opencode", "web", "--hostname", "0.0.0.0", "--port", str(port)],
        cwd=WORKSPACE, env=env, stdout=log, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    with open(PID_FILE, "w") as f:
        f.write(str(proc.pid))

    deadline = time.time() + wait_seconds
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError("OpenCode 啟動失敗，請查看 /content/logs/opencode.log")
        if _port_open(port):
            return ("opencode" if password else None), password
        time.sleep(2)
    raise RuntimeError("OpenCode 在時間內沒有回應，請查看 /content/logs/opencode.log")


def stop_portal():
    if os.path.exists(PID_FILE):
        with open(PID_FILE) as f:
            pid = int(f.read().strip() or 0)
        try:
            os.killpg(pid, signal.SIGTERM)
        except (ProcessLookupError, PermissionError):
            pass
        os.remove(PID_FILE)


def cleanup():
    """關閉入口與 Strata，刪除工作區、對話紀錄與日誌。"""
    stop_portal()
    strata = shutil.which("strata") or os.path.expanduser("~/.local/bin/strata")
    try:
        subprocess.run([strata, "launch", "--stop"], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except FileNotFoundError:
        pass  # 找不到 strata 就略過；刪除執行階段時主機會一併銷毀
    for path in (WORKSPACE, OPENCODE_DATA, LOG_DIR):
        shutil.rmtree(path, ignore_errors=True)
'''
FILES['viewer.py'] = r'''"""在 Colab 裡直接預覽工作區的檔案，不必下載。

筆記本的「👀 檢視工作區檔案」格呼叫 show()：選檔案 → 顯示在格子下方。
- Word／PowerPoint 等：LibreOffice 轉 PDF 後逐頁顯示成圖片（版面最接近原檔）
- Excel／CSV：表格；PDF：逐頁圖片；圖片、Markdown、文字檔：直接顯示；HTML（互動圖表）：內嵌
內容只顯示在老師自己的瀏覽器；筆記本設為「不儲存輸出」，不會寫回雲端硬碟。
"""
import glob
import html
import io
import os
import shutil
import subprocess
import tempfile

WORKSPACE = "/content/工作區"
HIDDEN = {"AGENTS.md"}
MAX_PAGES = 20          # 超過的頁數不顯示，避免畫面太長
MAX_TEXT = 200_000      # 文字檔最多顯示的字數
PAGE_DPI = "80"

IMAGE_EXT = {".png", ".jpg", ".jpeg", ".gif", ".bmp", ".webp", ".tif", ".tiff"}
OFFICE_EXT = {".docx", ".doc", ".odt", ".rtf", ".pptx", ".ppt", ".odp"}
TABLE_EXT = {".xlsx", ".xls", ".ods", ".csv"}
TEXT_EXT = {".txt", ".json", ".py", ".log", ".tsv", ".xml", ".yaml", ".yml"}


def list_files():
    """工作區的檔案（含子資料夾），最新修改的排最前面。"""
    found = []
    for root, dirs, files in os.walk(WORKSPACE):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        for name in files:
            rel = os.path.relpath(os.path.join(root, name), WORKSPACE)
            if rel not in HIDDEN and not name.startswith((".", "~$")):
                found.append(rel)
    return sorted(found, key=lambda r: os.path.getmtime(os.path.join(WORKSPACE, r)), reverse=True)


def _show_pdf_pages(pdf):
    from IPython.display import Image, display
    tmp = tempfile.mkdtemp(prefix="viewer_")
    try:
        subprocess.run(["pdftoppm", "-png", "-r", PAGE_DPI, "-l", str(MAX_PAGES), pdf, os.path.join(tmp, "p")],
                       check=True, capture_output=True, timeout=180)
        pages = sorted(glob.glob(os.path.join(tmp, "p*.png")))
        for page in pages:
            display(Image(filename=page))
        if len(pages) == MAX_PAGES:
            print(f"（只顯示前 {MAX_PAGES} 頁，完整內容請下載）")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def _show_office(path):
    tmp = tempfile.mkdtemp(prefix="viewer_")
    try:
        # 獨立的 LibreOffice 設定檔，避免和 AI 同時轉檔時互搶
        subprocess.run(["soffice", f"-env:UserInstallation=file://{tmp}/profile", "--headless",
                        "--convert-to", "pdf", "--outdir", tmp, path],
                       check=True, capture_output=True, timeout=240)
        pdfs = glob.glob(os.path.join(tmp, "*.pdf"))
        if not pdfs:
            raise RuntimeError("LibreOffice 沒有轉出 PDF")
        _show_pdf_pages(pdfs[0])
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def _show_table(path, ext):
    import pandas as pd
    from IPython.display import Markdown, display
    if ext == ".csv":
        sheets = {"": pd.read_csv(path)}
    else:
        sheets = pd.read_excel(path, sheet_name=None)
    for name, df in sheets.items():
        if name:
            display(Markdown(f"**工作表：{name}**（{len(df)} 列）"))
        display(df)


def _show_image(path):
    from IPython.display import Image, display
    from PIL import Image as PILImage
    if os.path.splitext(path)[1].lower() in {".png", ".jpg", ".jpeg", ".gif"}:
        display(Image(filename=path))
        return
    buf = io.BytesIO()
    PILImage.open(path).convert("RGB").save(buf, format="PNG")
    display(Image(data=buf.getvalue()))


def _show_text(path, markdown=False):
    from IPython.display import HTML, Markdown, display
    with open(path, encoding="utf-8", errors="replace") as f:
        text = f.read(MAX_TEXT + 1)
    if len(text) > MAX_TEXT:
        text = text[:MAX_TEXT]
        print(f"（只顯示前 {MAX_TEXT} 字）")
    if markdown:
        display(Markdown(text))
    else:
        display(HTML(f"<pre style='white-space:pre-wrap'>{html.escape(text)}</pre>"))


def preview(rel):
    """顯示一個工作區檔案；rel 為相對於工作區的路徑。"""
    from IPython.display import HTML, display
    path = os.path.join(WORKSPACE, rel)
    ext = os.path.splitext(rel)[1].lower()
    print(f"📄 {rel}")
    if ext == ".pdf":
        _show_pdf_pages(path)
    elif ext in OFFICE_EXT:
        _show_office(path)
    elif ext in TABLE_EXT:
        _show_table(path, ext)
    elif ext in IMAGE_EXT:
        _show_image(path)
    elif ext in {".md", ".markdown"}:
        _show_text(path, markdown=True)
    elif ext in {".html", ".htm"}:
        with open(path, encoding="utf-8", errors="replace") as f:
            display(HTML(f.read()))
    elif ext in TEXT_EXT:
        _show_text(path)
    else:
        print("這種檔案無法在這裡預覽，請按「下載這個檔案」。")


def show():
    """顯示檔案選單與按鈕。"""
    import ipywidgets as widgets
    from IPython.display import display

    picker = widgets.Dropdown(description="檔案", layout=widgets.Layout(width="70%"))
    view_btn = widgets.Button(description="👀 顯示", button_style="primary")
    refresh_btn = widgets.Button(description="🔄 重新整理清單")
    download_btn = widgets.Button(description="⬇️ 下載這個檔案")
    out = widgets.Output()

    def refresh(_=None):
        files = list_files()
        picker.options = files
        if not files:
            with out:
                out.clear_output()
                print("工作區還沒有檔案。用對話框左下的「＋」附加檔案，或請 AI 產出後再按「重新整理清單」。")

    def view(_):
        with out:
            out.clear_output()
            if not picker.value:
                print("請先選一個檔案。")
                return
            try:
                preview(picker.value)
            except Exception as e:  # 預覽失敗只提示，不影響其他功能
                print(f"⚠️ 無法預覽（{type(e).__name__}: {e}），請改用下載。")

    def download(_):
        from google.colab import files
        with out:
            if picker.value:
                files.download(os.path.join(WORKSPACE, picker.value))

    view_btn.on_click(view)
    refresh_btn.on_click(refresh)
    download_btn.on_click(download)
    refresh()
    display(widgets.VBox([picker, widgets.HBox([view_btn, refresh_btn, download_btn]), out]))
'''
for name, content in FILES.items():
    with open(f'/content/core/{name}', 'w', encoding='utf-8') as f:
        f.write(content)
print('✅ 核心腳本已就緒：', ', '.join(FILES))

In [ ]:
#@title ③ 安裝 Strata 並下載、載入模型（約 10 分鐘；同時在背景安裝文件工具）
!bash /content/core/setup_tools.sh --background
!bash /content/core/setup_strata.sh {MODEL}

In [ ]:
#@title ④ 安裝 OpenCode，確認文件工具已就緒並自我檢查
!bash /content/core/setup_opencode.sh
!bash /content/core/setup_tools.sh --wait

In [ ]:
#@title ⑤ 開啟私人 AI 入口
import sys, importlib
sys.path.insert(0, '/content/core')
import portal
importlib.reload(portal)  # 重跑時讀到最新的核心腳本
from google.colab import output

model_id = portal.write_config()
portal.start()
print(f'模型：{model_id}')
print(portal.reasoning_check())
print('👇 對話框就在下方，直接打字即可（檔案用輸入框左下的「＋」附加；思考強度可在入口切換 off／low／medium／high）。')
print('（入口內嵌在這本筆記本裡，只有你這個瀏覽器、這個 Google 帳號看得到；另開分頁或換瀏覽器會是 404）')
# 內嵌顯示：另開視窗的方式會被瀏覽器擋（彈出視窗／不同分頁沒有 Colab 的存取權）
# path 直接開到工作區的新對話，老師不必自己新增專案
output.serve_kernel_port_as_iframe(portal.PORTAL_PORT, path=portal.entry_path(), height=850)

In [ ]:
#@title ⏹ 啟動完成（「全部執行」會停在這一格，這是正常的）
# 用例外讓「全部執行」停在這裡，下方的檢視／下載／清理格就不會被自動執行。
# 不用核取方塊：Colab 會把勾選狀態存進筆記本，下次「全部執行」就會誤觸清理（實測發生過）。
class 啟動完成(Exception):
    def _render_traceback_(self):
        return []

print('✅ 啟動完成。下方的「檢視」「下載」「清理」格，需要時再各自按左邊的 ▶。')
raise 啟動完成()

## 📁 檢視、下載與清理

「全部執行」會在上一格停下，以下三格**只有你按 ▶ 才會執行**。

- 檔案請用對話框左下的「＋」附加。你附加的、AI 產出的檔案都在主機的「工作區」資料夾。
- 「檢視」可以直接在這裡看 Word、Excel、PowerPoint、PDF、圖片等，不必下載。
- 這本筆記本設定為**不儲存輸出**：檢視的內容只顯示在你的瀏覽器，不會存回雲端硬碟。

In [ ]:
#@title 👀 檢視工作區檔案（按 ▶，選檔案後按「顯示」）
import sys, importlib
sys.path.insert(0, '/content/core')
import viewer
importlib.reload(viewer)
viewer.show()

In [ ]:
#@title ⬇️ 下載結果（按 ▶ 把工作區打包成 zip）
import os, zipfile
from google.colab import files
ws = '/content/工作區'
names = [n for n in sorted(os.listdir(ws)) if n != 'AGENTS.md'] if os.path.isdir(ws) else []
if not names:
    print('工作區是空的，還沒有檔案可以下載。')
else:
    zip_path = '/content/處理結果.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, _, fnames in os.walk(ws):
            for fn in fnames:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, ws)
                if rel != 'AGENTS.md':
                    z.write(full, rel)
    print(f'✅ 已打包 {len(names)} 個項目，開始下載…')
    files.download(zip_path)

In [ ]:
#@title 🧹 清理（用完按 ▶：關閉 AI、刪除所有上傳檔案與對話紀錄）
import sys, os
sys.path.insert(0, '/content/core')
import portal
portal.cleanup()
if os.path.exists('/content/處理結果.zip'):
    os.remove('/content/處理結果.zip')
print('✅ 已關閉 AI、刪除工作區、對話紀錄與日誌。')
print('👉 最後請按選單「執行階段 → 中斷連線並刪除執行階段」，主機才會銷毀並停止扣運算單元。')